# bronze → silver: 年齢各歳・男女別人口(050008_toukeika_032)

SageMaker Unified Studio の JupyterLab(Python 3 カーネル)で上から順に実行する。
SQL は Athena(ワークグループ `jaws-festa-2026`)で流す。

先に `landing_to_bronze.ipynb` で bronze に書き出してあること。

| silver テーブル | 役割 | 読み元の bronze テーブル |
| --- | --- | --- |
| `silver_nenreikakusai` | 全期間を正しく読める本番用 | 1980〜2015 は `bronze_nenreikakusai`(128列定義)、2020 は `bronze_nenreikakusai_127` を、ブロックごとに見出しが一致する定義で読む |
| `silver_nenreikakusai_128` | 列ずれを見せる比較用 | `bronze_nenreikakusai_128` を位置で読んだまま型だけ付ける |
| `silver_nenreikakusai_127` | 列ずれを見せる比較用 | `bronze_nenreikakusai_127` を位置で読んだまま型だけ付ける |

bronze の3テーブルは同じ場所のファイルを読む。`bronze_nenreikakusai` と `_128` は定義も同じなので、
定義・件数・集計値が一致することを書き込む前に確かめる。

値の扱い(決定済み):

| 値 | silver での扱い |
| --- | --- |
| 人数の列 | `try_cast(... AS bigint)` で long にする |
| `-`(2020ブロックの40か所) | NULL(0にはしない)。件数を記録する |
| マイナスの値 | 推計値としてそのまま残す。件数を記録する |
| `年齢不詳`(2020ブロック) | NULL。2020年は国勢調査の不詳補完値で、年齢不詳が各歳に振り分け済みのため |

手順:

1. 見出しの確認 — bronze の各 CSV の見出しが、使う定義(Glue の列コメント)と一致するか
2. 書き込む前の確認(bronze を読むだけ) — 型変換できない値 / 合計の検算 / キー重複 / 年とブロック / 年の抜け / コードと年月日 / 男女別の3行 / 件数。前年比・検算を飛ばした行・市町村名の変化は記録だけ
   図で目視する — 5歳階級の推移 / 人口ピラミッド / `-`・マイナスの分布 / 総数の推移(止めない)
3. 書き込み — silver の3本を truncate(`DELETE FROM`)してから `INSERT`(何度実行しても同じ結果になる)
4. 書き込み後の確認 — 件数と NULL の数

確認で1つでも問題があれば、silver には何も書かない。

In [ ]:
# ── 設定 ──────────────────────────────────────────────────────────────────
import os

# True のあいだは確認だけして書き込まない。結果を見てから False にして再実行する。
DRY_RUN = True

REGION = "ap-northeast-1"
WORKGROUP = "jaws-festa-2026"
BUCKET_PREFIX = "jaws-festa-2026"
BRONZE_PREFIX = "akita/bronze/population_by_age/"

BRONZE_DB = '"awsdatacatalog"."jaws_festa_2026_akita"'
SILVER_DB = '"s3tablescatalog/jaws-festa-2026-silver"."akita"'

# 数値にならなくても許す値(NULL にする)。これ以外が出たら止める。
NULL_TOKENS = {"-"}

# 識別列。これ以外は人数なので bigint(Iceberg の long)にする。
STRING_COLUMNS = ["nengetsubi_code", "nengetsubi", "shichoson_code", "shichoson", "danjo"]

In [ ]:
import time
import boto3

session = boto3.Session(profile_name=os.environ.get("AWS_PROFILE"), region_name=REGION)
s3 = session.client("s3")
glue = session.client("glue")
athena = session.client("athena")
account_id = session.client("sts").get_caller_identity()["Account"]
BUCKET = f"{BUCKET_PREFIX}-{account_id}-{REGION}-an"


def run(sql: str, timeout: int = 300) -> list[list[str | None]]:
    """Athena で SQL を1本流して、見出し行を除いた結果を返す。"""
    qid = athena.start_query_execution(QueryString=sql, WorkGroup=WORKGROUP)["QueryExecutionId"]
    deadline = time.time() + timeout
    while True:
        st = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]
        if st["State"] in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        if time.time() > deadline:
            athena.stop_query_execution(QueryExecutionId=qid)
            raise TimeoutError(f"{timeout}秒で終わらなかったので止めた: {qid}")
        time.sleep(1)
    if st["State"] != "SUCCEEDED":
        raise RuntimeError(f"{st['State']}: {st.get('StateChangeReason')}\n{sql[:500]}")
    rows = []
    for page in athena.get_paginator("get_query_results").paginate(QueryExecutionId=qid):
        rows += [[c.get("VarCharValue") for c in r["Data"]] for r in page["ResultSet"]["Rows"]]
    return rows[1:] if rows and sql.lstrip().upper().startswith(("SELECT", "WITH")) else rows


def columns_of(table: str) -> list[dict]:
    t = glue.get_table(DatabaseName="jaws_festa_2026_akita", Name=table)["Table"]
    return t["StorageDescriptor"]["Columns"]


COLS_BRONZE = columns_of("bronze_nenreikakusai")
COLS_128 = columns_of("bronze_nenreikakusai_128")
COLS_127 = columns_of("bronze_nenreikakusai_127")
NAMES_BRONZE = [c["Name"] for c in COLS_BRONZE]
NAMES_128 = [c["Name"] for c in COLS_128]
NAMES_127 = [c["Name"] for c in COLS_127]

# bronze_nenreikakusai は bronze_nenreikakusai_128 と同じ定義のはず(列名・型・元の見出しコメント)
def spec(cols):
    return [(c["Name"], c["Type"], c.get("Comment", "")) for c in cols]

assert spec(COLS_BRONZE) == spec(COLS_128), "bronze_nenreikakusai の定義が bronze_nenreikakusai_128 と違う"
print(f"bucket: {BUCKET}")
print(f"bronze 定義: bronze_nenreikakusai {len(NAMES_BRONZE)}列 / _128 {len(NAMES_128)}列 / _127 {len(NAMES_127)}列")

## 1. 見出しの確認

bronze のテーブルは列を**位置**で読むので、CSV の見出しと定義がずれていても気づけない。
ここで各 CSV の1行目(見出し)を読み、Glue の列コメント(元の日本語見出し)と突き合わせて、
ブロックごとにどちらの定義が正しいかを決める。どちらとも一致しないブロックがあれば止める。

In [ ]:
import re

def expected_header(cols: list[dict]) -> list[str]:
    # コメントは「年齢不詳(2020ブロックには存在しない)」のように注記付きのことがあるので、括弧の前だけ使う
    return [c.get("Comment", "").split("(")[0] for c in cols]

EXP_128, EXP_127 = expected_header(COLS_128), expected_header(COLS_127)

keys = sorted(
    o["Key"]
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=BRONZE_PREFIX)
    for o in page.get("Contents", [])
    if o["Key"].endswith(".csv")
)
assert keys, f"bronze が空: s3://{BUCKET}/{BRONZE_PREFIX}  先に landing_to_bronze.ipynb を実行する"

BLOCK_DEF = {}  # block -> "128" / "127"
for key in keys:
    block = re.search(r"_(\d{4})\.csv$", key).group(1)
    head = s3.get_object(Bucket=BUCKET, Key=key, Range="bytes=0-65535")["Body"].read()
    # 改行の手前までをデコードする(途中で切ると多バイト文字が割れるため)
    header = head.split(b"\n", 1)[0].rstrip(b"\r").decode("utf-8-sig").split(",")
    header = [h.strip() for h in header]
    if header == EXP_128:
        BLOCK_DEF[block] = "128"
    elif header == EXP_127:
        BLOCK_DEF[block] = "127"
    else:
        diff = next((i for i, (a, b) in enumerate(zip(header, EXP_128)) if a != b), None)
        raise SystemExit(f"{block}: 見出しがどちらの定義とも一致しない(列数 {len(header)}、128列定義と最初に違う位置 {diff})")
    print(f"  {block}: {len(header)}列 -> {BLOCK_DEF[block]}列定義で読む")

OLD_BLOCKS = sorted(b for b, d in BLOCK_DEF.items() if d == "128")
NEW_BLOCKS = sorted(b for b, d in BLOCK_DEF.items() if d == "127")

## 2. 書き込む前の確認

silver に入れる SELECT をここで組み立てて、その結果に対して確認する。

| 確認 | 問題とみなす条件 |
| --- | --- |
| 型変換 | `NULL_TOKENS`(`-`)以外の、数値にならない値がある |
| 合計の検算 | 総数 ≠ 各歳+100歳以上+年齢不詳 / 男女計 ≠ 男+女 / 5歳階級 ≠ 各歳の合計(NULL を含む行は除く) |
| キー重複 | 年月日コード × 市町村コード × 男女別 が2行以上ある |
| 件数 | silver に入る行数と bronze の行数が違う / bronze の3テーブル(`bronze_nenreikakusai` / `_128` / `_127`)で行数が違う |
| bronze 3テーブルの整合 | `bronze_nenreikakusai` と `_128` の定義(列名・型・コメント)が違う / 数値にならない値・マイナスの件数が違う |
| 年とブロック | 行の年が、ファイルのブロックの範囲(そのブロックの先頭年〜次のブロックの前年。最後のブロックは上限なし)に入っていない |
| 年の抜け | 最初の年から最後の年までに、行が1つも無い年がある |
| コードと年月日 | 年月日コードの先頭4桁と、年月日の年が違う |
| 男女別の3行 | 年月日コード × 市町村コードごとに、男女計・男・女が1行ずつそろっていない |

次は問題にせず、記録だけする。

| 記録 | 内容 |
| --- | --- |
| マイナス・`-` の件数 | 型変換の確認で数えたもの |
| 5歳階級の前年比(秋田県・男女計) | いちばん大きく減った・増えたところ。列がずれると2019→2020の90〜94歳が −73% になる。止める基準は値の幅を見てから決める |
| 検算を飛ばした行 | NULL(`-`)を含むため合計の検算をしていない行の数 |
| 市町村名の変化 | 同じ市町村コードで名前が2つ以上あるもの(村→町など) |

これらはすべて bronze を読むだけで、silver には触らない。
県の注記どおり、市町村の合計と県の値は一致しないので、その検算はしない。

In [ ]:
BLOCK = "regexp_extract(\"$path\", '_([0-9]{4})\\.csv$', 1)"

def typed(col: str, src: str | None = None) -> str:
    """bronze の string 列を silver の型にする式。src が None なら NULL。"""
    if col in STRING_COLUMNS:
        return f"{src or col} AS {col}"
    if src is None:
        return f"CAST(NULL AS bigint) AS {col}"
    return f"try_cast(trim({src}) AS bigint) AS {col}"

def in_list(blocks: list[str]) -> str:
    return ", ".join(f"'{b}'" for b in blocks) or "NULL"

# bronze のテーブルと、それぞれを正しく読めるブロック(128列定義は 1980〜2015、127列定義は 2020)
# 本番用の読み元は bronze_nenreikakusai(128列定義)と bronze_nenreikakusai_127 の2つ。
# bronze_nenreikakusai_128 は bronze_nenreikakusai と同じ定義・同じファイルなので、
# 本番用には使わず(二重に数えない)、整合の確認と比較用 silver にだけ使う。
CANONICAL_SOURCES = [
    ("bronze_nenreikakusai", NAMES_128, OLD_BLOCKS),
    ("bronze_nenreikakusai_127", NAMES_127, NEW_BLOCKS),
]
ALL_BRONZE = [
    ("bronze_nenreikakusai", NAMES_128, OLD_BLOCKS),
    ("bronze_nenreikakusai_128", NAMES_128, OLD_BLOCKS),
    ("bronze_nenreikakusai_127", NAMES_127, NEW_BLOCKS),
]

# 本番用: ブロックごとに見出しが一致する定義から、列名で対応させて読む
canonical_parts = []
for table, names, blocks in CANONICAL_SOURCES:
    if blocks:
        # 127列定義には age_unknown が無い -> NULL(不詳補完値のため)
        canonical_parts.append(
            "SELECT " + ", ".join(typed(c, c if c in names else None) for c in NAMES_128)
            + f" FROM {BRONZE_DB}.{table} WHERE {BLOCK} IN ({in_list(blocks)})"
        )
SELECT_CANONICAL = "\nUNION ALL\n".join(canonical_parts)

# 比較用: bronze を位置で読んだまま型だけ付ける(列ずれをそのまま残す)
SELECT_128 = "SELECT " + ", ".join(typed(c, c) for c in NAMES_128) + f" FROM {BRONZE_DB}.bronze_nenreikakusai_128"
SELECT_127 = "SELECT " + ", ".join(typed(c, c) for c in NAMES_127) + f" FROM {BRONZE_DB}.bronze_nenreikakusai_127"

TARGETS = {
    "silver_nenreikakusai": (SELECT_CANONICAL, NAMES_128),
    "silver_nenreikakusai_128": (SELECT_128, NAMES_128),
    "silver_nenreikakusai_127": (SELECT_127, NAMES_127),
}

In [ ]:
problems = []
report = {}

# ── 型変換できない値 / マイナス(元の bronze の文字列で数える) ──────────────
def raw_value_counts(table: str, names: list[str], where: str = "") -> list[list[str]]:
    nums = [c for c in names if c not in STRING_COLUMNS]
    arr = "ARRAY[" + ", ".join(nums) + "]"
    return run(f"""
        SELECT CASE WHEN try_cast(trim(v) AS bigint) IS NULL THEN coalesce(trim(v), '<NULL>')
                    ELSE 'negative' END AS kind,
               count(*)
        FROM {BRONZE_DB}.{table} CROSS JOIN UNNEST({arr}) AS t(v)
        WHERE {where or 'TRUE'}
          AND (try_cast(trim(v) AS bigint) IS NULL OR try_cast(trim(v) AS bigint) < 0)
        GROUP BY 1 ORDER BY 2 DESC""")

# bronze の3テーブルそれぞれを、正しく読めるブロックだけで数える
counts_by_table = {}
for table, names, blocks in ALL_BRONZE:
    counts = {}
    if blocks:
        for kind, n in raw_value_counts(table, names, f"{BLOCK} IN ({in_list(blocks)})"):
            counts[kind] = counts.get(kind, 0) + int(n)
    counts_by_table[table] = counts
report["value_counts_by_table"] = counts_by_table

# 本番用の読み元(bronze_nenreikakusai + bronze_nenreikakusai_127)の合計
canon_counts = {}
for table, _, _ in CANONICAL_SOURCES:
    for kind, n in counts_by_table[table].items():
        canon_counts[kind] = canon_counts.get(kind, 0) + n
report["value_counts"] = canon_counts
print("数値にならない値・マイナスの件数:", canon_counts)
bad = {k: v for k, v in canon_counts.items() if k not in NULL_TOKENS and k != "negative"}
if bad:
    problems.append(f"想定外の数値にならない値: {bad}")

# bronze_nenreikakusai と _128 は同じ定義で同じファイルを読むので、集計も一致するはず
if counts_by_table["bronze_nenreikakusai"] != counts_by_table["bronze_nenreikakusai_128"]:
    problems.append("bronze_nenreikakusai と bronze_nenreikakusai_128 で、数値にならない値・マイナスの件数が違う: "
                    f"{counts_by_table['bronze_nenreikakusai']} / {counts_by_table['bronze_nenreikakusai_128']}")

# ── 合計の検算(本番用の SELECT に対して) ────────────────────────────────────
ages = [f"age_{i}" for i in range(100)]
sum_ages = " + ".join(ages + ["age_100_plus", "coalesce(age_unknown, 0)"])
cls_checks = " OR ".join(
    f"age_cls_{k*5}_{k*5+4} <> (" + " + ".join(f"age_{k*5+j}" for j in range(5)) + ")" for k in range(20)
)
(tot_bad, cls_bad), = run(f"""
    SELECT count_if(sosu <> ({sum_ages})),
           count_if({cls_checks})
    FROM ({SELECT_CANONICAL})""")
mf_bad, = run(f"""
    WITH s AS ({SELECT_CANONICAL})
    SELECT count(*)
    FROM s t JOIN s m ON m.nengetsubi_code = t.nengetsubi_code AND m.shichoson_code = t.shichoson_code AND m.danjo = '男（人）'
             JOIN s f ON f.nengetsubi_code = t.nengetsubi_code AND f.shichoson_code = t.shichoson_code AND f.danjo = '女（人）'
    WHERE t.danjo = '男女計（人）' AND t.sosu <> m.sosu + f.sosu""")[0]
report["identity_mismatch"] = {"sosu": int(tot_bad), "age_cls": int(cls_bad), "danjo": int(mf_bad)}
print("合計の検算で合わない行:", report["identity_mismatch"])
if any(report["identity_mismatch"].values()):
    problems.append(f"合計の検算が合わない: {report['identity_mismatch']}")

# ── キー重複 ────────────────────────────────────────────────────────────────
dup, = run(f"""
    SELECT count(*) FROM (
      SELECT nengetsubi_code, shichoson_code, danjo FROM ({SELECT_CANONICAL})
      GROUP BY 1, 2, 3 HAVING count(*) > 1)""")[0]
report["duplicate_keys"] = int(dup)
print("キー重複:", dup)
if int(dup):
    problems.append(f"キー重複が {dup} 件")

# ── bronze のキー列(ブロックごとに正しい定義で読む) ─────────────────────────
key_parts = []
for table, _, blocks in CANONICAL_SOURCES:
    if blocks:
        key_parts.append(
            f"SELECT {BLOCK} AS blk, nengetsubi_code, nengetsubi, shichoson_code, shichoson, danjo "
            f"FROM {BRONZE_DB}.{table} WHERE {BLOCK} IN ({in_list(blocks)})"
        )
BRONZE_KEYS = "\nUNION ALL\n".join(key_parts)

# ── 年とブロック: 先頭年 <= 年 < 次のブロックの先頭年(最後のブロックは上限なし) ──
all_blocks = sorted(BLOCK_DEF)
upper = {b: (int(all_blocks[i + 1]) if i + 1 < len(all_blocks) else 9999) for i, b in enumerate(all_blocks)}
upper_case = "CASE blk " + " ".join(f"WHEN '{b}' THEN {u}" for b, u in upper.items()) + " END"
(out_of_block,), = run(f"""
    SELECT count_if(NOT (CAST(substr(nengetsubi, 1, 4) AS integer) >= CAST(blk AS integer)
                         AND CAST(substr(nengetsubi, 1, 4) AS integer) < {upper_case}))
    FROM ({BRONZE_KEYS})""")
report["out_of_block_rows"] = int(out_of_block)
print("ブロックの範囲外の年の行:", out_of_block)
if int(out_of_block):
    problems.append(f"ブロックの範囲外の年の行が {out_of_block} 件")

# ── 年の抜け ──────────────────────────────────────────────────────────────
years = sorted(int(y) for (y,) in run(f"SELECT DISTINCT substr(nengetsubi, 1, 4) FROM ({BRONZE_KEYS})"))
missing = sorted(set(range(years[0], years[-1] + 1)) - set(years))
report["years"] = {"first": years[0], "last": years[-1], "missing": missing}
print(f"年: {years[0]}〜{years[-1]}  抜け: {missing or 'なし'}")
if missing:
    problems.append(f"行が無い年がある: {missing}")

# ── 年月日コードと年月日 ────────────────────────────────────────────────────
(code_mismatch,), = run(f"""
    SELECT count_if(substr(nengetsubi_code, 1, 4) <> substr(nengetsubi, 1, 4)) FROM ({BRONZE_KEYS})""")
report["code_date_mismatch"] = int(code_mismatch)
print("年月日コードと年月日が合わない行:", code_mismatch)
if int(code_mismatch):
    problems.append(f"年月日コードと年月日が合わない行が {code_mismatch} 件")

# ── 男女別の3行 ────────────────────────────────────────────────────────────
(not_three,), = run(f"""
    SELECT count(*) FROM (
      SELECT nengetsubi_code, shichoson_code
      FROM ({BRONZE_KEYS})
      GROUP BY 1, 2
      HAVING count(*) <> 3
          OR count(DISTINCT danjo) <> 3
          OR count_if(danjo NOT IN ('男女計（人）', '男（人）', '女（人）')) > 0)""")
report["incomplete_danjo_groups"] = int(not_three)
print("男女別が3行そろっていない組:", not_three)
if int(not_three):
    problems.append(f"男女別が3行そろっていない組が {not_three} 件")

# ── 記録: 検算を飛ばした行(NULL を含む) ────────────────────────────────────
(skip_sosu, skip_cls), = run(f"""
    SELECT count_if((sosu <> ({sum_ages})) IS NULL),
           count_if(({cls_checks}) IS NULL)
    FROM ({SELECT_CANONICAL})""")
report["identity_skipped_rows"] = {"sosu": int(skip_sosu), "age_cls": int(skip_cls)}
print("記録: 検算を飛ばした行(NULL を含む):", report["identity_skipped_rows"])

# ── 記録: 5歳階級の前年比(秋田県・男女計) ──────────────────────────────────
cls_names = [f"age_cls_{k*5}_{k*5+4}" for k in range(20)]
yoy = run(f"""
    WITH s AS (
      SELECT substr(nengetsubi, 1, 4) AS y, {", ".join(cls_names)}
      FROM ({SELECT_CANONICAL})
      WHERE shichoson = '秋田県' AND danjo = '男女計（人）'),
    u AS (
      SELECT y, cls, v FROM s
      CROSS JOIN UNNEST(ARRAY[{", ".join(f"'{c}'" for c in cls_names)}],
                        ARRAY[{", ".join(cls_names)}]) AS t(cls, v))
    SELECT y, cls, round(CAST(v AS double) / lag(v) OVER (PARTITION BY cls ORDER BY y) - 1, 4) AS r
    FROM u ORDER BY y, cls""")
yoy = [(y, c, float(r)) for y, c, r in yoy if r is not None]
lo, hi = min(yoy, key=lambda x: x[2]), max(yoy, key=lambda x: x[2])
report["age_cls_yoy_pref"] = {"min": {"year": lo[0], "cls": lo[1], "ratio": lo[2]},
                              "max": {"year": hi[0], "cls": hi[1], "ratio": hi[2]}}
print(f"記録: 5歳階級の前年比(秋田県) 最小 {lo[0]} {lo[1]} {lo[2]:+.1%} / 最大 {hi[0]} {hi[1]} {hi[2]:+.1%}")

# ── 記録: 同じ市町村コードで名前が変わったもの ────────────────────────────
renamed = run(f"""
    SELECT shichoson_code, array_join(array_sort(array_agg(DISTINCT shichoson)), ' / ')
    FROM ({BRONZE_KEYS})
    GROUP BY 1 HAVING count(DISTINCT shichoson) > 1 ORDER BY 1""")
report["renamed_municipalities"] = {c: n for c, n in renamed}
print("記録: 名前が変わった市町村コード:", report["renamed_municipalities"] or "なし")

# ── 件数 ────────────────────────────────────────────────────────────────────
rows_by_table = {t: int(run(f"SELECT count(*) FROM {BRONZE_DB}.{t}")[0][0]) for t, _, _ in ALL_BRONZE}
bronze_rows = rows_by_table["bronze_nenreikakusai"]
canon_rows = int(run(f"SELECT count(*) FROM ({SELECT_CANONICAL})")[0][0])
report["rows"] = {"bronze": bronze_rows, "bronze_by_table": rows_by_table, "silver_nenreikakusai": canon_rows}
print("件数:", report["rows"])
if bronze_rows == 0:
    problems.append("bronze が0行")
if len(set(rows_by_table.values())) != 1:
    problems.append(f"bronze の3テーブルで件数が違う: {rows_by_table}")
if bronze_rows != canon_rows:
    problems.append(f"件数が合わない: bronze {bronze_rows} / silver {canon_rows}")

print()
print("確認結果:", "OK" if not problems else "NG")
for p in problems:
    print("  -", p)

## 2-2. bronze を可視化して確認する

数値の確認では「どこが・どう」おかしいかが見えにくいので、bronze を図にして目で確かめる。
すべて bronze を読むだけで、silver には触らない。⑤⑥は `silver_nenreikakusai` に書き込む内容(`SELECT_CANONICAL`)を図にするので、書き込む前に中身を見られる。問題があっても止めない(見て判断する)。

| 図 | 見ること |
| --- | --- |
| ① 5歳階級の推移(秋田県・男女計) | 見出しで読んだ値(本番用)と、128列定義で位置読みした値を重ねる。2020年以降で位置読みだけが跳ねれば列ずれ |
| ② 人口ピラミッド 2019 / 2020 | 位置読みの2020年は形が崩れる(若い階級に1つ上の階級の値が入る) |
| ③ `-`・マイナスの分布 | ブロック × 列のどこに集まっているか。高齢の列・小さい町村に偏っていれば想定どおり |
| ④ 秋田県の総数の推移 | 不自然な段差が無いか(国勢調査で基準が変わる年に小さな段差が出るのは想定どおり) |
| ⑤ 数値カラムごとの NULL とマイナス(`silver_nenreikakusai`) | 数値の123列それぞれの NULL とマイナスの数。NULL は `age_unknown`(2020ブロックの468行)と `-` の位置だけのはず |
| ⑥ 年 × 各歳のヒートマップ(`silver_nenreikakusai`、秋田県・男女計) | `age_0`〜`age_100_plus` を年ごとに並べる。上は人数、下は前年からの変化率。縦・横に不自然な筋や急な色の変化が無いか |

図のラベルは、日本語フォントが無い環境でも文字化けしないよう英字にしている。

In [ ]:
import matplotlib.pyplot as plt

CLS = [f"age_cls_{k*5}_{k*5+4}" for k in range(20)]
CLS_LABEL = [f"{k*5}-{k*5+4}" for k in range(20)]
PREF = "shichoson = '秋田県' AND danjo = '男女計（人）'"

def to_int(v):
    try:
        return int(v)
    except (TypeError, ValueError):
        return None

def pref_series(select_sql: str, cols: list[str]) -> dict[int, list[int | None]]:
    rows = run(f"SELECT substr(nengetsubi, 1, 4), {', '.join(cols)} FROM ({select_sql}) WHERE {PREF} ORDER BY 1")
    return {int(r[0]): [to_int(v) for v in r[1:]] for r in rows}

canon = pref_series(SELECT_CANONICAL, CLS + ["sosu", "age_unknown"])
pos128 = pref_series(SELECT_128, CLS + ["sosu", "age_unknown"])
years_all = sorted(canon)

# ── ① 5歳階級の推移: 本番用(見出しで読む) vs 128列定義の位置読み ──────────────
show = ["age_cls_0_4", "age_cls_45_49", "age_cls_85_89", "age_cls_90_94", "age_cls_95_99"]
fig, axes = plt.subplots(1, len(show), figsize=(4 * len(show), 3.2), sharex=True)
for ax, col in zip(axes, show):
    i = CLS.index(col)
    ax.plot(years_all, [canon[y][i] for y in years_all], label="by header (silver)", lw=2)
    ax.plot(years_all, [pos128[y][i] for y in years_all], label="by position (128 cols)", ls="--")
    ax.axvline(2020, color="gray", lw=0.8)
    ax.set_title(col)
axes[0].legend(fontsize=8)
fig.suptitle("(1) Akita pref, both sexes: 5-year age classes")
fig.tight_layout()
plt.show()

In [ ]:
# ── ② 人口ピラミッド 2019 / 2020(男・女、5歳階級) ─────────────────────────────
def pyramid(select_sql: str, year: int) -> tuple[list, list]:
    rows = run(f"""
        SELECT danjo, {', '.join(CLS)} FROM ({select_sql})
        WHERE shichoson = '秋田県' AND substr(nengetsubi, 1, 4) = '{year}'
          AND danjo IN ('男（人）', '女（人）')""")
    d = {r[0]: [to_int(v) or 0 for v in r[1:]] for r in rows}
    return d.get("男（人）", [0] * 20), d.get("女（人）", [0] * 20)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, (title, sql, year) in zip(axes, [
    ("2019 by header", SELECT_CANONICAL, 2019),
    ("2020 by header", SELECT_CANONICAL, 2020),
    ("2020 by position (128 cols)", SELECT_128, 2020),
]):
    m, f = pyramid(sql, year)
    ax.barh(range(20), [-x for x in m], color="tab:blue", label="male")
    ax.barh(range(20), f, color="tab:red", label="female")
    ax.set_yticks(range(20), CLS_LABEL)
    ax.set_title(title)
    ax.axvline(0, color="black", lw=0.5)
axes[0].legend(fontsize=8)
fig.suptitle("(2) Population pyramid, Akita pref")
fig.tight_layout()
plt.show()

In [ ]:
# ── ③ "-"・マイナスの分布(ブロック × 列、本番用の読み方) ─────────────────────
dist = {}
for table, names, blocks in CANONICAL_SOURCES:
    if not blocks:
        continue
    nums = [c for c in names if c not in STRING_COLUMNS]
    rows = run(f"""
        SELECT {BLOCK} AS blk, col,
               count_if(trim(v) = '-') AS dash,
               count_if(try_cast(trim(v) AS bigint) < 0) AS neg
        FROM {BRONZE_DB}.{table}
        CROSS JOIN UNNEST(ARRAY[{", ".join(f"'{c}'" for c in nums)}], ARRAY[{", ".join(nums)}]) AS t(col, v)
        WHERE {BLOCK} IN ({in_list(blocks)})
        GROUP BY 1, 2""")
    for blk, col, dash, neg in rows:
        dist[(blk, col)] = (int(dash), int(neg))

blocks_sorted = sorted(BLOCK_DEF)
num_cols = [c for c in NAMES_128 if c not in STRING_COLUMNS]
fig, axes = plt.subplots(2, 1, figsize=(16, 5), sharex=True)
for ax, k, title in ((axes[0], 0, '"-" (becomes NULL)'), (axes[1], 1, "negative values (kept)")):
    grid = [[dist.get((b, c), (0, 0))[k] for c in num_cols] for b in blocks_sorted]
    im = ax.imshow(grid, aspect="auto", cmap="Reds")
    ax.set_yticks(range(len(blocks_sorted)), blocks_sorted)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, pad=0.01)
step = 10
axes[1].set_xticks(range(0, len(num_cols), step), num_cols[::step], rotation=45, ha="right", fontsize=8)
fig.suptitle("(3) Where '-' and negative values are (block x column)")
fig.tight_layout()
plt.show()

In [ ]:
# ── ④ 秋田県の総数の推移 ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(years_all, [canon[y][CLS.__len__()] for y in years_all], marker="o", ms=3)
for b in sorted(BLOCK_DEF):
    ax.axvline(int(b), color="gray", lw=0.6, ls=":")
ax.set_title("(4) Akita pref total population (sosu); dotted = census base years")
ax.ticklabel_format(axis="y", style="plain")
fig.tight_layout()
plt.show()

In [ ]:
# ── ⑤ silver_nenreikakusai の数値カラムごとの NULL とマイナス ──────────────────
# silver に書く内容(SELECT_CANONICAL)をそのまま使う。書き込み前に確かめられる。
import numpy as np

NUM_COLS = [c for c in NAMES_128 if c not in STRING_COLUMNS]   # 数値の123列
rows = run(f"""
    SELECT col, count_if(v IS NULL), count_if(v < 0)
    FROM ({SELECT_CANONICAL})
    CROSS JOIN UNNEST(ARRAY[{", ".join(f"'{c}'" for c in NUM_COLS)}], ARRAY[{", ".join(NUM_COLS)}]) AS t(col, v)
    GROUP BY col""")
stat = {c: (int(n), int(m)) for c, n, m in rows}
nulls = [stat[c][0] for c in NUM_COLS]
negs = [stat[c][1] for c in NUM_COLS]

fig, axes = plt.subplots(2, 1, figsize=(16, 5), sharex=True)
axes[0].bar(range(len(NUM_COLS)), nulls, color="tab:orange")
axes[0].set_title("NULL count per numeric column")
axes[1].bar(range(len(NUM_COLS)), negs, color="tab:red")
axes[1].set_title("negative count per numeric column")
marks = ["sosu", "age_0", "age_50", "age_99", "age_100_plus", "age_unknown", "age_cls_0_4", "age_cls_95_99"]
axes[1].set_xticks([NUM_COLS.index(m) for m in marks], marks, rotation=45, ha="right", fontsize=8)
fig.suptitle(f"(5) silver_nenreikakusai: numeric columns ({len(NUM_COLS)} cols) - NULL and negative values")
fig.tight_layout()
plt.show()
print(f"NULL の合計 {sum(nulls)}(うち age_unknown {stat['age_unknown'][0]})/ マイナスの合計 {sum(negs)}")

In [ ]:
# ── ⑥ 年 × 各歳のヒートマップ(silver_nenreikakusai、秋田県・男女計) ─────────────
AGE_COLS = [f"age_{i}" for i in range(100)] + ["age_100_plus"]
rows = run(f"""
    SELECT substr(nengetsubi, 1, 4), {", ".join(AGE_COLS)}
    FROM ({SELECT_CANONICAL}) WHERE {PREF} ORDER BY 1""")
yrs = [int(r[0]) for r in rows]
M = np.array([[np.nan if v is None else float(v) for v in r[1:]] for r in rows]).T   # 行=年齢, 列=年
with np.errstate(divide="ignore", invalid="ignore"):
    D = M[:, 1:] / M[:, :-1] - 1                  # 前年からの変化率
D = np.where(np.isfinite(D), D, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
im = axes[0].imshow(M, origin="lower", aspect="auto", cmap="viridis",
                    extent=[yrs[0] - 0.5, yrs[-1] + 0.5, -0.5, 100.5])
axes[0].set_title("population by single age")
fig.colorbar(im, ax=axes[0])
im = axes[1].imshow(D, origin="lower", aspect="auto", cmap="RdBu_r", vmin=-0.3, vmax=0.3,
                    extent=[yrs[1] - 0.5, yrs[-1] + 0.5, -0.5, 100.5])
axes[1].set_title("change from previous year (clipped to +-30%)")
fig.colorbar(im, ax=axes[1])
for ax in axes:
    ax.set_xlabel("year")
    ax.set_ylabel("age (100 = 100+)")
    for b in sorted(BLOCK_DEF):
        ax.axvline(int(b), color="white", lw=0.6, ls=":")
fig.suptitle("(6) silver_nenreikakusai: age_0 ... age_100_plus by year (Akita pref, both sexes); dotted = census base years")
fig.tight_layout()
plt.show()

## 3. silver に書き込む

確認がすべて通ったときだけ書く。2段階で行う。

1. **truncate** — silver の3本を全部空にし、0行になったことを確かめる
2. **INSERT** — 3本に書き込む

Athena の Iceberg(S3 Tables)には `TRUNCATE TABLE` が無いので、条件なしの `DELETE FROM` で
空にする。Iceberg のスナップショットとして記録されるので、消す前の状態はタイムトラベルで見られる。
何度実行しても同じ結果になる。truncate から INSERT が終わるまでの間、silver は空になる。

In [ ]:
if problems:
    raise SystemExit(f"確認に失敗したので silver には書かない: {problems}")

# ── (1) truncate: 3本とも空にする ────────────────────────────────────────────
for table in TARGETS:
    target = f"{SILVER_DB}.{table}"
    if DRY_RUN:
        print(f"[dry-run] truncate {table}(DELETE FROM)")
        continue
    run(f"DELETE FROM {target}")
    (n,), = run(f"SELECT count(*) FROM {target}")
    assert int(n) == 0, f"truncate 後に {table} が {n} 行残っている"
    print(f"truncated {target}")

# ── (2) INSERT ──────────────────────────────────────────────────────────────
for table, (select_sql, names) in TARGETS.items():
    target = f"{SILVER_DB}.{table}"
    insert_sql = f"INSERT INTO {target} ({', '.join(names)})\n{select_sql}"
    if DRY_RUN:
        print(f"[dry-run] INSERT {table}({len(names)}列)")
        continue
    run(insert_sql)
    print(f"wrote {target}")

## 4. 書き込み後の確認

In [ ]:
if not DRY_RUN:
    for table in TARGETS:
        (n, unknown_null, last_null), = run(f"""
            SELECT count(*), count_if(age_unknown IS NULL), count_if(age_cls_95_99 IS NULL)
            FROM {SILVER_DB}.{table}""") if table != "silver_nenreikakusai_127" else run(f"""
            SELECT count(*), NULL, count_if(age_cls_95_99 IS NULL)
            FROM {SILVER_DB}.{table}""")
        print(f"{table:<28} 行 {n:>5}  age_unknown NULL {unknown_null}  age_cls_95_99 NULL {last_null}")
    (n,), = run(f"SELECT count(*) FROM {SILVER_DB}.silver_nenreikakusai")
    assert int(n) == report["rows"]["bronze"], f"silver_nenreikakusai の行数 {n} が bronze {report['rows']['bronze']} と違う"
    print("書き込み後の確認 OK")

## SQL を確認したいとき

次のセルを実行すると、silver に流す `INSERT` 文をそのまま表示する。
Athena のクエリエディタ(ワークグループ `jaws-festa-2026`)に貼れば同じことができる。

In [ ]:
print("-- truncate")
for table in TARGETS:
    print(f"DELETE FROM {SILVER_DB}.{table};")
print()
for table, (select_sql, names) in TARGETS.items():
    print(f"-- {table}")
    print(f"INSERT INTO {SILVER_DB}.{table} ({', '.join(names)})\n{select_sql};\n")